# Part I walkthrough — where the compression comes from

**Run this alongside the Part I slides.** Every claim made from the podium is
measured here, in order. You are encouraged to change the numbers and break things.

The question Part I answers is narrow and specific:

> A field on a $2^n$-point grid is a vector with $2^n$ entries. Why should a handful
> of small matrices be able to stand in for it, and when do they stop being able to?

We will answer it by measurement rather than assertion.

In [ ]:
import time

import matplotlib.pyplot as plt
import numpy as np

import qtade_tn as tn

np.set_printoptions(precision=3, suppress=True)
plt.rcParams.update({"figure.figsize": (9, 3.2), "axes.grid": True,
                     "grid.alpha": 0.3, "font.size": 10})

## 1. The wall

Nothing subtle here. Count the unknowns, in three dimensions, per field, per timestep.

In [ ]:
for n in (6, 8, 10, 12):
    N = 2 ** n
    print(f"{N:5d}^3 grid: {N**3:>18,d} unknowns   "
          f"({N**3 * 8 / 2**30:>10.1f} GiB per field, float64)")

print("\nDNS needs ~Re^(9/4) degrees of freedom in 3D:")
for Re in (1e3, 1e5, 1e7):
    print(f"  Re = {Re:>8.0e}:  {Re**2.25:>10.2e} dof")

An aircraft at cruise sits near $\mathrm{Re}\sim10^7$. Exascale hardware moves that
wall by about two orders of magnitude. The gap is sixteen.

So: no discretisation fixes this. Finite differences, volumes and elements change the
constant in front, never the exponent. Every classical escape — adaptive meshes,
multigrid, spectral methods, POD — works by exploiting *structure* in the solution.
Tensor networks are one more of those, not a different kind of thing.

## 2. A physical field is not an arbitrary vector

This is the whole bet, and it is falsifiable. Take a smooth field and a noise field of
the same size. Reshape each into a matrix by cutting the index in half, and look at the
singular values.

**Predict before you run it:** how fast do you expect each to decay?

In [ ]:
n = 12
x = np.linspace(0, 1, 2 ** n, endpoint=False)
smooth = np.exp(-3 * x) + np.sin(4 * np.pi * x)
noise = np.random.default_rng(0).standard_normal(2 ** n)

fig, ax = plt.subplots(1, 2)
for label, v in [("smooth", smooth), ("noise", noise)]:
    M = v.reshape(2 ** (n // 2), -1)
    s = np.linalg.svd(M, compute_uv=False)
    ax[0].semilogy(s / s[0], label=label)
ax[0].set(xlabel="singular value index", ylabel="$\\sigma_k/\\sigma_0$",
          title="decay across the middle cut")
ax[0].legend()
ax[1].plot(x, smooth, label="smooth")
ax[1].plot(x, noise, lw=0.4, alpha=0.7, label="noise")
ax[1].set(xlabel="x", title="the two fields")
ax[1].legend()
plt.tight_layout()

Machine precision in about ten singular values, versus no decay at all. That gap *is*
the compression. Everything else in this course is bookkeeping around it.

## 3. TT-SVD: turning the decay into a data structure

Reshape, SVD, split off one core, repeat. `tt_svd` is fifteen lines; read it.

In [ ]:
cores = tn.qtt_from_vector(smooth, eps=1e-10)
print("bond dimensions:", tn.tt_ranks(cores))
print(f"stored floats:   {tn.tt_size(cores):,d}   (dense: {smooth.size:,d})")
print(f"compression:     {smooth.size / tn.tt_size(cores):.1f}x")
print(f"relative error:  "
      f"{np.linalg.norm(tn.qtt_to_vector(cores) - smooth) / np.linalg.norm(smooth):.2e}")

## 4. The quantics trick, and the claim that matters

The reshape above was not arbitrary. Writing the grid index in binary,

$$x = \sum_{k=1}^{n} i_k 2^{-k},$$

makes core $k$ responsible for one length scale: core 1 says which half of the domain,
core 2 which quarter, and so on. A bond then measures the coupling *between* scales.

The claim to test: **rank tracks smoothness, not resolution.** Doubling the resolution
should add one core and leave the bond dimensions alone.

In [ ]:
print(f"{'n':>4} {'N':>9} {'max chi':>8} {'params':>9} {'dense':>12} {'ratio':>10}")
for n in range(6, 21, 2):
    xs = np.linspace(0, 1, 2 ** n, endpoint=False)
    f = np.exp(-3 * xs) + np.sin(4 * np.pi * xs)
    c = tn.qtt_from_vector(f, eps=1e-10)
    print(f"{n:>4} {2**n:>9,d} {max(tn.tt_ranks(c)):>8} {tn.tt_size(c):>9,d} "
          f"{2**n:>12,d} {2**n / tn.tt_size(c):>9.0f}x")

The `max chi` column is flat. That is the entire quantics promise, and it is the reason
a $2^{30}$-cell mesh is not absurd.

## 5. Which functions are cheap, and why

Some ranks are exact and known in closed form. The exponential is rank 1 because it
*factorises over the digits*:

$$e^{a\sum_k i_k 2^{-k}} = \prod_k e^{a i_k 2^{-k}}.$$

A sine is rank 2 because it is a sum of two such exponentials. A polynomial of degree
$p$ is rank $p+1$.

**Predict first.** Rank the seven functions below from cheapest to most expensive.
At least one of them will not do what you expect.

In [ ]:
n = 16
xs = np.linspace(0, 1, 2 ** n, endpoint=False)
tests = {
    "exp(-3x)": np.exp(-3 * xs),
    "sin(8 pi x)": np.sin(8 * np.pi * xs),
    "x^3 - x": xs ** 3 - xs,
    "gaussian, width 0.05": np.exp(-((xs - 0.5) / 0.05) ** 2),
    "step at x = 1/3": (xs > 1 / 3).astype(float),
    "sin(1/(x + 0.01))": np.sin(1 / (xs + 0.01)),
    "white noise": np.random.default_rng(1).standard_normal(2 ** n),
}
for name, f in tests.items():
    c = tn.qtt_from_vector(f, eps=1e-8)
    print(f"{name:>22}:  max chi = {max(tn.tt_ranks(c)):>4}   params = {tn.tt_size(c):>7,d}")

### The step function is rank 2. Sit with that for a moment.

A discontinuity is the textbook enemy of spectral methods, and the intuition "sharp
edge, slow singular-value decay, huge rank" is so natural that it is worth saying
clearly that **it is false here**.

The reason is that quantics does not see a jump; it sees a comparison. The indicator
$\mathbb{1}[i > m]$ is decided digit by digit, most significant first, by an automaton
with two states: *still equal to $m$ so far*, and *already decided*. Two states, bond
dimension 2 — whatever the resolution, and wherever the jump sits.

What *is* expensive is a feature that is not aligned with the binary index: an
oscillation the coarse cores cannot resolve, or — in two dimensions — a curved
boundary. Both couple many scales at once, and coupling scales is precisely what a bond
has to pay for.

In [ ]:
# The same lesson in two dimensions: a straight edge versus a circle.
import qtade_quimb as qq

for m in (6, 8, 10):
    N = 2 ** m
    xx = np.linspace(0, 1, N, endpoint=False)
    XX, YY = np.meshgrid(xx, xx, indexing="ij")
    half_plane = (XX > 0.3).astype(float)
    disc = (((XX - 0.5) ** 2 + (YY - 0.5) ** 2) < 0.25 ** 2).astype(float)
    print(f"N = {N:>5}:  half-plane chi = "
          f"{max(tn.tt_ranks(qq.from_grid(half_plane, eps=1e-8))):>4},   "
          f"disc chi = {max(tn.tt_ranks(qq.from_grid(disc, eps=1e-8))):>4}")

The half-plane is free at every resolution. The disc is not, and it gets worse as the
grid refines: the circle cuts across every length scale at once, so every bond has to
carry information about where it is. **That** is the geometry problem, and Part II
spends a whole subsection buying it off with a smoothed mask.

## 6. Operators: a derivative is a small automaton

A shift by one grid point is *addition of 1 to a binary number*. Adding one propagates
a carry from the least significant digit, and a carry is one bit of state. One bit of
state, two states, bond dimension 2. Nothing is fitted and nothing is truncated.

In [ ]:
n = 4
S = tn.qtt_shift(n, +1)
print("shift MPO ranks:", tn.mpo_ranks(S))
print(np.round(tn.mpo_full(S)).astype(int)[:6, :6], "  <- ones on the subdiagonal\n")

L = tn.qtt_laplacian(n, dx=1.0)
print("Laplacian MPO ranks:", tn.mpo_ranks(L), " (2 + 2 + 1 = 5 before rounding)")
print(np.round(tn.mpo_full(L)).astype(int)[:5, :5])

Rank 3 for a second derivative is not a coincidence: a three-point stencil is a sum of
three shifts, and a shift is a two-state carry automaton. Small automaton, small bond
dimension. The dense $2^n \times 2^n$ matrix is never formed, at any point.

## 7. A PDE solver, in two lines

$$\partial_t u = \alpha \partial_x^2 u, \qquad u^{k+1} = (\mathbb{1} + \alpha \Delta t\,L)\,u^k$$

Two operations per step: apply an MPO, then **round**. Watch the bond dimension.

In [ ]:
n, alpha = 16, 1.0
h = 2.0 ** -n
dt = 0.4 * h ** 2 / alpha                     # explicit stability: dt <= h^2/2 alpha
xs = np.linspace(0, 1, 2 ** n, endpoint=False)

L = tn.qtt_laplacian(n, dx=h)
step = tn.mpo_round(tn.mpo_add(tn.mpo_identity(n), tn.mpo_scale(L, alpha * dt)), 1e-13)

u = tn.qtt_from_vector(np.exp(-((xs - 0.5) / 0.05) ** 2), eps=1e-10)
history = [max(tn.tt_ranks(u))]
snapshots = {0: tn.qtt_to_vector(u)}

t0 = time.perf_counter()
for k in range(1, 601):
    u = tn.tt_round(tn.mpo_apply(step, u), eps=1e-8)
    history.append(max(tn.tt_ranks(u)))
    if k in (200, 600):
        snapshots[k] = tn.qtt_to_vector(u)
print(f"600 steps on a {2**n:,d}-point grid in {time.perf_counter() - t0:.2f} s")

fig, ax = plt.subplots(1, 2)
for k, v in snapshots.items():
    ax[0].plot(xs, v, label=f"step {k}")
ax[0].set(xlim=(0.25, 0.75), xlabel="x", title="diffusion")
ax[0].legend()
ax[1].plot(history)
ax[1].set(xlabel="step", ylabel="max bond dimension", title="$\\chi(t)$")
plt.tight_layout()

$\chi$ falls. Diffusion smooths, smoothness is cheap, so the representation gets
*cheaper* as the solution evolves. If $\chi$ had climbed instead, the suspects are the
encoding, the boundary treatment, or a rounding tolerance that is too loose.

## 8. What happens if you forget to round

The classic first mistake. Each MPO application multiplies the bond dimension by
$\chi_{\text{MPO}} = 3$, so the rank grows like $3^k$ — and the *memory* like $3^{2k}$,
because a core is $\chi \times 2 \times \chi$. Start from a rank-2 state (a sine) on a
smaller grid so that this cell finishes rather than taking the kernel with it.

In [ ]:
n_small = 10
h_s = 2.0 ** -n_small
xs_s = np.linspace(0, 1, 2 ** n_small, endpoint=False)
L_s = tn.qtt_laplacian(n_small, dx=h_s)
step_s = tn.mpo_round(
    tn.mpo_add(tn.mpo_identity(n_small), tn.mpo_scale(L_s, 0.4 * h_s ** 2)), 1e-13)

u = tn.qtt_from_vector(np.sin(2 * np.pi * xs_s), eps=1e-12)
chi0 = max(tn.tt_ranks(u))
print(f"{'step':>5} {'chi (no rounding)':>18} {'3^k * chi_0':>13} {'memory':>12}")
print(f"{0:>5} {chi0:>18,d} {chi0:>13,d} {tn.tt_size(u) * 8 / 2**20:>10.2f} MiB")
for k in range(1, 7):
    u = tn.mpo_apply(step_s, u)
    print(f"{k:>5} {max(tn.tt_ranks(u)):>18,d} {3**k * chi0:>13,d} "
          f"{tn.tt_size(u) * 8 / 2**20:>10.2f} MiB")

rounded = tn.tt_round(u, eps=1e-10)
print(f"\nafter one rounding: chi = {max(tn.tt_ranks(rounded))}, "
      f"{tn.tt_size(rounded) * 8 / 2**10:.1f} KiB — "
      f"the extra rank was never information, only bookkeeping.")

It tracks $3^k$ exactly, and the memory tracks $9^k$. Two more steps and this cell
would exhaust the machine; meanwhile the *actual* rank of the solution never exceeded
a handful.

**Rounding is not an optimisation — it is the algorithm.** Everything from here on is
a variation on "apply something, then round".

---
Next: Part II, where "apply" is no longer enough and we have to *solve*.